# 🎬 SubtitleStudioX — Edición Google Colab (GPU Cloud)

Este cuaderno traslada la lógica de **SubtitleStudioX** a la nube para aprovechar la potencia de una GPU T4 gratuita. Procesa múltiples capítulos o archivos pesados sin necesidad de instalar nada en tu PC.

---

## 🛠️ ¿Qué incluye esta versión?
* **GPU en la nube**: WhisperX `large-v3` + `float16` (5-10x más rápido que CPU)
* **Alineación milimétrica**: Wav2Vec2 para sincronización perfecta
* **Inmune a errores**: Escritor SRT nativo en Python
* **Procesamiento por lotes**: Bucle continuo hasta que decidas salir
* **Ingeniería de audio**: Filtros `firequalizer` + `dynaudnorm` + traducción latina con `Helsinki-NLP/opus-mt-en-es`  

---

## 🚀 Guía de Uso

### 1️⃣ Ejecuta la Celda 1 (instalación, 3-5 min)
### 2️⃣ Ejecuta la Celda 2 (procesamiento continuo)

> 💻 ¿Prefieres tu PC? Usa [SubtitleStudio](https://github.com/KingEdhard/SubtitleStudio)

In [ ]:
# ============================================
# CELDA 1: Configuración del Entorno
# ============================================
import os
import subprocess

usuario = "KingEdhard"
proyecto = "SubtitleStudioX"
url_git = f"https://github.com/{usuario}/{proyecto}.git"

print(f"📦 Clonando proyecto desde: {url_git}")
with open('instalador_seguro.sh', 'w') as f:
    f.write(f"#!/bin/bash\nrm -rf {proyecto} github.com\ngit clone {url_git}\ncd {proyecto}\npip install -r requirements.txt --quiet\n")

!chmod +x instalador_seguro.sh && ./instalador_seguro.sh

print("📦 Instalando motor WhisperX...")
!pip install "git+https://github.com/m-bain/whisperx.git" --quiet

print("⚙️ Configurando compatibilidad de FFmpeg para Linux...")
!apt-get install ffmpeg -y > /dev/null
%cd /content/{proyecto}
!mkdir -p bin
!ln -sf $(which ffmpeg) bin/ffmpeg.exe
!ln -sf $(which ffprobe) bin/ffprobe.exe

print("\n✅ [CELDA 1 COMPLETADA] El entorno está listo.")

In [ ]:
# ============================================
# CELDA 2: Procesamiento en Bucle Continuo
# ============================================
import os
import sys
from google.colab import files

# Asegurar la ruta correcta dentro del proyecto
if os.path.exists('/content/SubtitleStudioX'):
    %cd /content/SubtitleStudioX
elif os.path.exists('SubtitleStudioX'):
    %cd SubtitleStudioX

# FORZAR VERSIONES COMPATIBLES DE PYTORCH
print("⚙️ Instalando PyTorch compatible con WhisperX (toma 1-2 min)...")
!pip uninstall torch torchvision torchaudio -y --quiet 2>/dev/null
!pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --quiet --force-reinstall 2>/dev/null
print("✅ PyTorch estabilizado.")

# PROTECCIÓN ANTI-FALLOS: Instalar WhisperX si no existe
print("🔍 Verificando instalación de WhisperX...")
try:
    import whisperx
    print("✅ WhisperX ya está instalado y listo para usar.")
except ImportError:
    print("⚠️ WhisperX no encontrado. Forzando instalación global...")
    !pip install "git+https://github.com/m-bain/whisperx.git" --quiet
    print("✅ WhisperX fijado correctamente.")

# 🔧 Silenciar warning de sacremoses
print("🔧 Instalando sacremoses para evitar warnings...")
try:
    import sacremoses
    print("✅ sacremoses ya instalado.")
except ImportError:
    !pip install sacremoses --quiet 2>/dev/null
    print("✅ sacremoses instalado.")

print("\n[⚙️] Generando main.py optimizado...")

codigo_definitivo = '''import sys
import os
import torch
from tqdm import tqdm
from src.components.extraccion import extraer_audio_mejorado
from src.components.traduccion import traducir_srt
from src.components.muxer import incrustar_subtitulos

def transcribir_con_whisperx(wav_path):
    import whisperx
    device, compute_type = "cuda", "float16"
    
    print("\\n🚀 [IA] Iniciando WhisperX con el mejor modelo (Large-v3)...")
    model = whisperx.load_model("large-v3", device, compute_type=compute_type, language="en")
    audio = whisperx.load_audio(wav_path)
    result = model.transcribe(audio, batch_size=16)
    
    print("🎯 [IA] Aplicando alineación fonética milimétrica (Wav2Vec2)...")
    model_a, metadata = whisperx.load_align_model(language_code="en", device=device)
    result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)
    
    srt_path = wav_path.replace("_dialogos_mejorados.wav", "_en.srt")
    
    def format_time(t):
        h, m, s, ms = int(t // 3600), int((t % 3600) // 60), int(t % 60), int((t % 1) * 1000)
        return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"
        
    with open(srt_path, "w", encoding="utf-8") as f:
        for idx, segment in enumerate(result["segments"], 1):
            start = format_time(segment["start"])
            end = format_time(segment["end"])
            f.write(f"{idx}\\n{start} --> {end}\\n{segment['text'].strip()}\\n\\n")
            
    print(f"✔ Subtítulos milimétricos generados con éxito: {srt_path}")
    return srt_path

def main():
    print("\\n=== SUBTITLESTUDIOX + WHISPERX (Versión Ultra Precisa) ===")
    print("Ingeniería de audio + Alineación milimétrica + Traducción latino\\n")
    
    formatos_video = ('.mp4', '.mkv', '.avi', '.mov', '.flv', '.wmv')
    archivos = [f for f in os.listdir('.') if f.lower().endswith(formatos_video)]
    if not archivos:
        print("❌ No se encontró ningún archivo de vídeo en la carpeta actual.")
        return
        
    total = len(archivos)
    print(f"📂 {total} vídeo(s) detectado(s).")
    for i, video in enumerate(archivos, 1):
        print(f"\\n{'='*60}\\n🎬 Procesando: {video}\\n{'='*60}")
        try:
            wav = extraer_audio_mejorado(video)
            if not wav: continue
            srt_ingles = transcribir_con_whisperx(wav)
            if not srt_ingles:
                if os.path.exists(wav): os.remove(wav)
                continue
            print("\\n🌎 Traduciendo subtítulos a español latino con tu Helsinki-NLP/opus-mt-en-es...")
            srt_espanol = traducir_srt(srt_ingles)
            ruta_final = incrustar_subtitulos(video, srt_ingles, srt_espanol if srt_espanol else srt_ingles)
            if ruta_final:
                print(f"\\n🏁 ¡Vídeo completado con éxito!: {ruta_final}")
            if os.path.exists(wav): os.remove(wav)
        except Exception as e:
            print(f"\\n❌ Error procesando {video}: {e}")
            if 'wav' in locals() and os.path.exists(wav): os.remove(wav)
            continue

if __name__ == '__main__':
    main()'''

with open("main.py", "w", encoding="utf-8") as f:
    f.write(codigo_definitivo)

print("✅ main.py generado.")
print("-" * 60)

# BUCLE INFINITO DE CARGA Y PROCESAMIENTO
while True:
    formatos_video = ('.mp4', '.mkv', '.avi', '.mov', '.flv', '.wmv')
    videos_existentes = [f for f in os.listdir('.') if f.lower().endswith(formatos_video)]
    
    if videos_existentes:
        print(f"\nℹ️ {len(videos_existentes)} video(s) en el servidor:")
        for v in videos_existentes:
            print(f"   🔹 {v}")
        print("🚀 Procesando...\n")
    else:
        print("\n🎬 NO HAY VIDEOS. Sube capítulos o películas:")
        uploaded = files.upload()
        if not uploaded:
            print("\n👋 ¿Terminar sesión?")
            respuesta = input("   Escribe 'S' para salir o Enter para reintentar: ")
            if respuesta.upper() == 'S':
                print("🛑 Sesión finalizada.")
                break
            else:
                continue
    
    !python main.py
    
    print("-" * 60)
    print("✅ Lote completado. Descarga del panel 📁")
    print("-" * 60)
    
    print("\n🔄 ¿CARGAR NUEVO VIDEO O PELI?")
    respuesta = input("   Enter para subir más | 'S' para salir: ")
    if respuesta.upper() == 'S':
        print("🎉 ¡Sesión completada! Descarga tus videos 📁")
        break
    else:
        for v in videos_existentes:
            try:
                os.remove(v)
                print(f"🗑 Limpiando: {v}")
            except:
                pass
        print("\n📤 Listo para nuevos archivos...\n")